<a href="https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Age and content decline

The paper reports that growing pages were younger than declining pages: about 185 days versus 228 days on average, while average word count was almost identical. The label comes from the observed trend direction, comparing pages classified as growing or declining.

My methodology question is whether this relationship remains visible under a grouped validation design and whether age adds useful ranking information beyond other observed page features. The comparison is observational, so it does not establish that age itself causes decline.

### Finding 2 — The CTR cliff

The paper reports that click-through rate changes substantially across search-position tiers, with lower-ranked pages generally receiving much lower CTR. The outcome is observed CTR from search performance data.

My methodology question is whether position remains a useful signal in my smaller anonymized dataset and whether the relationship is stable enough to support a content-review decision. I would treat this as directional evidence rather than proof that changing position alone causes CTR to change.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Paper-finding checks on the starter dataset

import os
import subprocess
import pandas as pd
import numpy as np

REPO_DIR = "/content/flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

csv_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found: {csv_path}")

df = pd.read_csv(csv_path)

# Finding 1: age by trend direction
age_summary = (
    df.groupby("trend_direction")["content_age_days"]
      .mean()
      .round(1)
)

word_summary = (
    df.groupby("trend_direction")["word_count"]
      .mean()
      .round(1)
)

print("Mean content age by trend direction:")
print(age_summary)

print("\nMean word count by trend direction:")
print(word_summary)


# Finding 2: CTR by position tier
ctr_summary = (
    df.groupby("position_tier")["ctr"]
      .mean()
      .sort_values(ascending=False)
      .round(4)
)

print("\nMean CTR by position tier:")
print(ctr_summary)


Mean content age by trend direction:
trend_direction
down      236.2
flat      245.9
new       238.7
stable    295.4
up        288.5
Name: content_age_days, dtype: float64

Mean word count by trend direction:
trend_direction
down      3221.8
flat      2616.0
new       2382.2
stable    3347.2
up        2998.1
Name: word_count, dtype: float64

Mean CTR by position tier:
position_tier
top_3       1.4836
page_1      0.6525
striking    0.3232
page_3_5    0.2225
deep        0.1502
Name: ctr, dtype: float64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I will compare the model under a random row split and a client-grouped split. The grouped split is the more conservative validation because pages from the same client cannot appear in both training and testing. If performance falls under the grouped split, I will report that reduction rather than selecting the more favorable result. Precision@50 remains the main decision metric because the practical use case is prioritizing a small review queue.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Compare random split vs client-grouped split

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

work = df.copy()

# Target
work["is_declining"] = (
    work["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Features that are available without using the final trend outcome.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

# Explicitly exclude outcome/future-window fields.
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

numeric_features = [
    c for c in numeric_features
    if c in work.columns and c not in leakage_fields
]

categorical_features = [
    c for c in categorical_features
    if c in work.columns and c not in leakage_fields
]

# Numeric matrix
X_num = (
    work[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Categorical matrix
X_cat = (
    work[categorical_features]
    .fillna("unknown")
    .astype(str)
)

X_cat = pd.get_dummies(
    X_cat,
    prefix=categorical_features,
    dtype=float
)

X = pd.concat(
    [
        X_num.reset_index(drop=True),
        X_cat.reset_index(drop=True)
    ],
    axis=1
)

y = work["is_declining"].reset_index(drop=True)


def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    return (
        temp.sort_values("score", ascending=False)
            .head(k)["actual"]
            .mean()
    )


def train_and_evaluate(train_idx, test_idx, split_name):

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    scores = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    p50 = precision_at_k(
        y.iloc[test_idx],
        scores,
        50
    )

    p100 = precision_at_k(
        y.iloc[test_idx],
        scores,
        100
    )

    auc = roc_auc_score(
        y.iloc[test_idx],
        scores
    )

    return {
        "split": split_name,
        "Precision@50": p50,
        "Precision@100": p100,
        "ROC-AUC": auc
    }


# ------------------------------------
# A. Random row split
# ------------------------------------
all_idx = np.arange(len(work))

random_train, random_test = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)


# ------------------------------------
# B. Client-grouped split
# ------------------------------------
clients = work["client_id"].fillna("unknown").astype(str)

unique_clients = clients.unique()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)

n_test_clients = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:n_test_clients]
)

group_test_mask = clients.isin(test_clients)

group_test = np.where(group_test_mask)[0]
group_train = np.where(~group_test_mask)[0]

# Verify no client overlap.
assert (
    set(clients.iloc[group_train])
    .isdisjoint(set(clients.iloc[group_test]))
)

results = pd.DataFrame([
    train_and_evaluate(
        random_train,
        random_test,
        "Random row split"
    ),
    train_and_evaluate(
        group_train,
        group_test,
        "Client-grouped split"
    )
])

print("Validation comparison:")
display(results.round(3))

print("\nClient overlap in grouped split: 0")

Validation comparison:


,split,Precision@50,Precision@100,ROC-AUC
0,Random row split,0.9,0.90,0.756
1,Client-grouped split,0.7,0.67,0.744



Client overlap in grouped split: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final feature set for fields derived from the target and for fields representing later performance windows. The target is trend_direction, so trend_direction and trend_pct must not be model inputs. I also exclude the last-30-day versus previous-30-day trend components because they directly encode the comparison used to define the outcome. The model should use information available independently of the final decline label.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3. Final leakage audit

# These are the fields that should never enter the final model.
known_leakage = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

final_features = set(
    numeric_features + categorical_features
)

detected_leakage = sorted(
    final_features.intersection(known_leakage)
)

print("Final feature count:", len(final_features))

print("\nKnown leakage fields found in final features:")
print(detected_leakage)

assert not detected_leakage, (
    f"Leakage detected: {detected_leakage}"
)


# Additional name-based future-window audit.
future_keywords = [
    "future",
    "post",
    "after",
    "next",
    "later"
]

future_named_fields = [
    field
    for field in final_features
    if any(
        keyword in field.lower()
        for keyword in future_keywords
    )
]

print("\nPossible future-named fields:")
print(future_named_fields)


# Product-flag audit.
# Note: 'age_tier_order' is NOT a product flag.
product_keywords = [
    "product_flag",
    "product_issue",
    "product_score",
    "product_action",
    "optimization_flag"
]

product_fields = [
    field
    for field in final_features
    if any(
        keyword in field.lower()
        for keyword in product_keywords
    )
]

print("\nPossible product-flag fields:")
print(product_fields)

assert not product_fields, (
    f"Possible product flags detected: {product_fields}"
)

print("\nLeakage audit: PASSED")
print("Target-derived fields: none")
print("Explicit future-window fields: none")
print("Product-flag fields: none")

Final feature count: 32

Known leakage fields found in final features:
[]

Possible future-named fields:
[]

Possible product-flag fields:
[]

Leakage audit: PASSED
Target-derived fields: none
Explicit future-window fields: none
Product-flag fields: none


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Safe research claim

In this anonymized dataset, the Random Forest produced a higher observed Precision@50 than the fixed baseline under the tested validation design. The result is directional evidence that combining multiple page-level signals can improve prioritization of potential content-refresh opportunities. It should be treated as decision-support, not as causal proof that any individual feature causes traffic decline or that the model predicts search-engine behavior.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 4. Claim audit summary

print("CLAIM AUDIT")
print("-" * 50)

print(
    "Observed: model performance was measured on a held-out test set."
)

print(
    "Directional: results indicate whether the ranking signal "
    "is useful in this dataset."
)

print(
    "Decision-support: scores prioritize pages for human review."
)

print(
    "Not claimed: causal effects, algorithmic search-engine proof, "
    "or guaranteed traffic improvement."
)

print("\nValidation table:")
display(results.round(3))

CLAIM AUDIT
--------------------------------------------------
Observed: model performance was measured on a held-out test set.
Directional: results indicate whether the ranking signal is useful in this dataset.
Decision-support: scores prioritize pages for human review.
Not claimed: causal effects, algorithmic search-engine proof, or guaranteed traffic improvement.

Validation table:


,split,Precision@50,Precision@100,ROC-AUC
0,Random row split,0.9,0.90,0.756
1,Client-grouped split,0.7,0.67,0.744


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.